In [2]:
# Imports
import sys
from pathlib import Path
import warnings

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.pategan.models import PATEGAN

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# ---------- Custom PATEGAN for CAR ----------
class PATEGAN(PATEGAN):
    def __init__(self):
        super().__init__(
            epsilon=5.0,     # relaxed privacy → better scores
            delta=1e-5,
            num_teachers=5,  # fewer teachers → more data per teacher
            niter=20000,      # you can change to 1000/8000 later
            batch_size=64,
            learning_rate=5e-4,
            lambda_gp=5.0,
            random_state=42,
        )

# ---------------- Preprocess data (CAR) ----------------
dataset_path = ROOT / "raw_data" / "shuttle.csv"
output_path = ROOT / "discretized_data" / "shuttle.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

# ---------------- Run Train/Test/Synthetic pipeline ----------------
input_csv = str(output_path)          # discretised CAR data
output_dir = str(ROOT / "sample_data" / "shuttle")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "shuttle" / "pategan")

model_preview = PATEGAN()
print("PATEGAN will train for:", model_preview.niter, "iterations")

pipeline = TrainTestSplitPipeline(model=PATEGAN)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\shuttle.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\shuttle.csv
PATEGAN will train for: 20000 iterations
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
0    0.785970
3    0.153491
4    0.056336
2    0.002953
1    0.000862
6    0.000216
5    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.785948
3    0.153534
4    0.056293
2    0.002931
1    0.000862
6    0.000259
5    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
Remapped y classes: [0, 1, 2, 3, 4, 5, 6] -> [0, 1, 2, 3, 4, 5, 6]
Loaded training data: X shape=(46400, 9), y shape=(46400,)
Training PATE-GAN with ε=5.0, δ=1e-05
Teachers: 5, Iterations: 20000


Training: 100%|██████████| 20000/20000 [10:52<00:00, 30.67it/s]


Training completed!

Generating 46400 synthetic samples...
[PATEGAN] Adding 4 dummy samples to cover classes: [0, 4, 5, 6]
Remapped y_test.csv to match synthetic data encoding
Saved x_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\shuttle\pategan\x_synth.csv
Saved y_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\shuttle\pategan\y_synth.csv
Saved metadata.json to C:\Users\Prabu\Downloads\Katabatic\synthetic\shuttle\pategan\metadata.json


C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\xgboost\training.py:199: UserWarning: [22:49:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results\shuttle\pategan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.5091
F1 Score: 0.5803

MLP:
Accuracy: 0.3517
F1 Score: 0.4766

RF:
Accuracy: 0.4172
F1 Score: 0.5423

XGBoost:
Accuracy: 0.2053
F1 Score: 0.2981
Train test split pipeline executed successfully.
